# Camada Gold

Monta o modelo dimensional de atendimento a partir das tabelas da Silver e grava as tabelas no schema `gold`. O modelo é um esquema estrela: a fato `fato_chamado` tem uma linha por chamado, e as dimensões descrevem motivo, canal, atendente, cliente, data, hora e status. Toda dimensão tem um membro "Não Informado", com chave -1, para onde apontam os chamados sem correspondência. No fim, consultas de validação conferem a integridade do modelo.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import date

# criação dos nomes/caminhos dos schemas de cada camada
catalog = "vcredit_medallion"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"


## Funções auxiliares

`mostrar` exibe o resultado de uma verificação quando há casos.

In [0]:
def mostrar(df_, descricao):
    if not df_.isEmpty():
        print(f"\n{descricao}")
        display(df_)

## Tabelas de origem

Lista as tabelas disponíveis na Silver.

In [0]:
tables = spark.sql(f"SHOW TABLES IN {silver_schema}")
display(tables)

## 1. dim_motivo

Origem: `silver.tb_base_motivos`. A SK é numerada pela ordem do `id_motivo`.

In [0]:
# partição única explícita: a dimensão é pequena, então juntar tudo numa partição não pesa
w_sk = Window.partitionBy(F.lit(1)).orderBy(F.col("id_motivo").cast("int"))

df_silver_motivo = spark.read.table(f"{silver_schema}.tb_base_motivos")

df_dim_motivo = (
    df_silver_motivo
    .withColumn("SK_motivo", F.row_number().over(w_sk).cast("bigint"))
    .select(
        F.col("SK_motivo"),
        F.col("id_motivo").cast("int").alias("id_motivo"),
        F.col("nome_motivo").cast("string").alias("nome"),
        F.col("categoria").cast("string").alias("categoria"),
        F.col("criticidade").cast("string").alias("criticidade"),
    )
    .orderBy("SK_motivo")
)

# ---------- membro "não informado" ----------
df_dim_motivo = df_dim_motivo.unionByName(
    spark.createDataFrame([(-1, "Não Informado")], "SK_motivo bigint, nome string"),
    allowMissingColumns=True,
).orderBy("SK_motivo")

display(df_dim_motivo)

df_dim_motivo.write.mode("overwrite").saveAsTable(f"{gold_schema}.dim_motivo")

## 2. dim_canal

Origem: `silver.tb_canais`. A Silver não traz a etapa do funil, então ela é derivada do nome do canal: URA e Chatbot são autosserviço, seguidos de atendimento inicial e atendimento especializado. `nivel_etapa` ordena as etapas na sequência do funil.

In [0]:
w_sk = Window.partitionBy(F.lit(1)).orderBy(F.col("id_canal").cast("int"))

df_silver_canais = spark.read.table(f"{silver_schema}.tb_canais")

# a silver não traz etapa_funil nem nivel_etapa: derivamos dos valores padronizados na Silver
k_nome_canal = F.lower(F.trim(F.col("nome_canal")))

etapa_funil = (
    F.when(k_nome_canal == "ura", F.lit("Autosserviço"))
    .when(k_nome_canal == "chatbot", F.lit("Autosserviço"))
    .when(k_nome_canal == "atendimento inicial", F.lit("Atendimento Inicial"))
    .when(k_nome_canal == "atendimento especializado", F.lit("Atendimento Especializado"))
    .otherwise(F.lit("Desconhecido"))
)

# ordem manual, para refletir a posição da etapa no funil de atendimento
nivel_etapa = (
    F.when(etapa_funil == "Autosserviço", F.lit(1))
    .when(etapa_funil == "Atendimento Inicial", F.lit(2))
    .when(etapa_funil == "Atendimento Especializado", F.lit(3))
    .otherwise(F.lit(None).cast("int"))
)

df_dim_canal = (
    df_silver_canais
    .withColumn("SK_canal", F.row_number().over(w_sk).cast("bigint"))
    .withColumn("etapa_funil", etapa_funil)
    .withColumn("nivel_etapa", nivel_etapa)
    .select(
        F.col("SK_canal"),
        F.col("id_canal").cast("int").alias("id_canal"),
        F.col("nome_canal").cast("string").alias("nome"),
        F.col("tipo_canal").cast("string").alias("tipo"),
        F.col("etapa_funil").cast("string").alias("etapa_funil"),
        F.col("nivel_etapa").cast("int").alias("nivel_etapa"),
        F.col("status_canal").cast("string").alias("status"),
    )
    .orderBy("SK_canal")
)

# ---------- membro "não informado" ----------
df_dim_canal = df_dim_canal.unionByName(
    spark.createDataFrame([(-1, "Não Informado")], "SK_canal bigint, nome string"),
    allowMissingColumns=True,
).orderBy("SK_canal")

display(df_dim_canal)

df_dim_canal.write.mode("overwrite").saveAsTable(f"{gold_schema}.dim_canal")

## 3. dim_atendente

Origem: `silver.tb_base_atendentes`.

In [0]:
w_sk = Window.partitionBy(F.lit(1)).orderBy(F.col("id_atendente").cast("int"))

df_silver_atendentes = spark.read.table(f"{silver_schema}.tb_base_atendentes")

df_dim_atendente = (
    df_silver_atendentes
    .withColumn("SK_atendente", F.row_number().over(w_sk).cast("bigint"))
    .select(
        F.col("SK_atendente"),
        F.col("id_atendente").cast("int").alias("id_atendente"),
        F.col("nome_atendente").cast("string").alias("nome"),
        F.col("nivel_atendimento").cast("int").alias("nivel_atendimento"),
        F.col("data_admissao").cast("date").alias("data_admissao"),
        F.col("data_desligamento").cast("date").alias("data_desligamento"),
        F.col("status_atendente").cast("string").alias("status"),
    )
    .orderBy("SK_atendente")
)

# ---------- membro "não informado" ----------
df_dim_atendente = df_dim_atendente.unionByName(
    spark.createDataFrame([(-1, "Não Informado")], "SK_atendente bigint, nome string"),
    allowMissingColumns=True,
).orderBy("SK_atendente")

display(df_dim_atendente)

df_dim_atendente.write.mode("overwrite").saveAsTable(f"{gold_schema}.dim_atendente")

## 4. dim_cliente

Origem: `silver.tb_clientes`. A idade é agrupada em faixas de 10 anos, com mais detalhe entre 18 e 84 anos, onde está a maior parte dos clientes.

In [0]:
# id_cliente (CPF) fica string: com 11 dígitos, a ordem de texto coincide com a numérica
w_sk = Window.partitionBy(F.lit(1)).orderBy(F.col("id_cliente"))

df_silver_clientes = spark.read.table(f"{silver_schema}.tb_clientes")

# idade já foi normalizada como inteiro na camada silver
idade = F.col("idade")

# maior quantidade de clientes se encontram na faixa entre 30-80 anos
# então buscamos maior granularidade por faixa de 10 anos entre 18 e 80 anos
faixa_idade = (
    F.when(idade.isNull() , F.lit("Não Informado"))
    .when(idade <= 24, F.lit("18-24"))
    .when(idade <= 34, F.lit("25-34"))
    .when(idade <= 44, F.lit("35-44"))
    .when(idade <= 54, F.lit("45-54"))
    .when(idade <= 64, F.lit("55-64"))
    .when(idade <= 74, F.lit("65-74"))
    .when(idade <= 84, F.lit("75-84"))
    .otherwise(F.lit("85+"))
)

df_dim_cliente = (
    df_silver_clientes
    .withColumn("SK_cliente", F.row_number().over(w_sk).cast("bigint"))
    .withColumn("faixa_idade", faixa_idade)
    .select(
        F.col("SK_cliente"),
        F.col("id_cliente").cast("string").alias("id_cliente"),
        F.col("nome").cast("string").alias("nome"),
        F.col("email").cast("string").alias("email"),
        F.col("regiao").cast("string").alias("regiao"),
        F.col("faixa_idade").cast("string").alias("faixa_idade"),
        F.col("data_cadastro").alias("data_cadastro"),
        F.expr("try_cast(dia_vencimento_fatura AS INT)").alias("dia_vencimento_fatura"),
    )
    .orderBy("SK_cliente")
)

# ---------- membro "não informado" ----------
df_dim_cliente = df_dim_cliente.unionByName(
    spark.createDataFrame([(-1, "Não Informado")], "SK_cliente bigint, nome string"),
    allowMissingColumns=True,
).orderBy("SK_cliente")


display(df_dim_cliente)

df_dim_cliente.write.mode("overwrite").saveAsTable(f"{gold_schema}.dim_cliente")

## 5. dim_data

Calendário gerado do primeiro dia do ano da data mais antiga até o último dia do ano da data mais recente encontradas em `silver.tb_chamados_hora`. A SK tem o formato `yyyyMMdd`.

In [0]:
df_silver_chamados_hora = spark.read.table(f"{silver_schema}.tb_chamados_hora")

# ---------- limites do calendário ----------
limites = df_silver_chamados_hora.agg(
    F.min(F.least("abertura_data", "inicio_data", "fim_data")).alias("min"),
    F.max(F.greatest("abertura_data", "inicio_data", "fim_data")).alias("max"),
).first()

if limites["min"] is None:
    raise ValueError("Nenhuma data válida encontrada em tb_chamados_hora")

data_inicio = date(limites["min"].year, 1, 1)
data_fim    = date(limites["max"].year, 12, 31)
print(f"Datas nos chamados: {limites['min']} a {limites['max']}")
print(f"Calendário gerado:  {data_inicio} a {data_fim}")

# ---------- calendário ----------
DIAS_SEMANA = ["Domingo", "Segunda-feira", "Terça-feira", "Quarta-feira",
               "Quinta-feira", "Sexta-feira", "Sábado"]          # dayofweek: 1 = domingo

df_calendario = spark.range(1).select(
    F.explode(F.sequence(F.lit(data_inicio), F.lit(data_fim), F.expr("INTERVAL 1 DAY"))).alias("data")
)

df_dim_data = df_calendario.select(
    (F.year("data") * 10000 + F.month("data") * 100 + F.dayofmonth("data")).cast("int").alias("SK_data"),
    F.col("data"),
    F.year("data").alias("ano"),
    F.month("data").alias("mes"),
    F.dayofmonth("data").alias("dia"),
    F.element_at(F.array(*[F.lit(d) for d in DIAS_SEMANA]), F.dayofweek("data")).alias("dia_semana"),
)

# ---------- membro "não informado" ----------
membro_desconhecido = spark.createDataFrame(
    [(-1, None, None, None, None, "Não Informado")],
    "SK_data int, data date, ano int, mes int, dia int, dia_semana string",
)
df_dim_data = df_dim_data.unionByName(membro_desconhecido).orderBy("SK_data")

display(df_dim_data)

df_dim_data.write.mode("overwrite").saveAsTable(f"{gold_schema}.dim_data")

## 6. dim_hora

Todos os minutos do dia, de 00:00 a 23:59. A SK tem o formato `hhmm`, por exemplo 923 para 09:23.

In [0]:
df_dim_hora = (
    spark.range(24 * 60)                                   # id = minuto do dia (0 a 1439)
    .withColumn("hora", (F.col("id") / 60).cast("int"))
    .withColumn("minuto", (F.col("id") % 60).cast("int"))
    .select(
        (F.col("hora") * 100 + F.col("minuto")).alias("SK_hora"),   # 09:23 -> 923
        "hora",
        "minuto",
    )
)

# ---------- membro "não informado" ----------
membro_desconhecido = spark.createDataFrame(
    [(-1, None, None)],
    "SK_hora int, hora int, minuto int",
)
df_dim_hora = df_dim_hora.unionByName(membro_desconhecido).orderBy("SK_hora")

display(df_dim_hora)

df_dim_hora.write.mode("overwrite").saveAsTable(f"{gold_schema}.dim_hora")

## 7. Junção de chamados com horários

Junta `silver.tb_chamados` com `silver.tb_chamados_hora` pelo `id_chamado` e verifica o resultado antes de montar a fato.

In [0]:
df_silver_chamados = spark.read.table(f"{silver_schema}.tb_chamados")

# lado da hora: id_cliente renomeado + marcador para saber se o chamado achou par no join
df_hora = (
    df_silver_chamados_hora
    .withColumnRenamed("id_cliente", "id_cliente_hora")
    .withColumn("_tem_hora", F.lit(True))
)

df_chamados_com_hora = df_silver_chamados.join(df_hora, "id_chamado", "left")

n_chamados = df_silver_chamados.count()
n_join = df_chamados_com_hora.count()
print(f"Linhas em tb_chamados: {n_chamados} | após o join: {n_join}")
if n_join > n_chamados:
    print("  -> há id_chamado repetido em tb_chamados_hora: o join duplicou chamados")
else:
    print("  -> ok, não há duplicação")

### 7.1 id_cliente nas duas tabelas

Compara o `id_cliente` de `tb_chamados` com o de `tb_chamados_hora` para o mesmo chamado.

In [0]:
c, h = F.col("id_cliente"), F.col("id_cliente_hora")
df_comp = df_chamados_com_hora.withColumn(
    "status_id_cliente",
    F.when(F.col("_tem_hora").isNull(), "sem registro em chamados_hora")
     .when(c.isNull() & h.isNull(), "nulo nos dois")
     .when(c.isNull(), "nulo só em chamados")
     .when(h.isNull(), "nulo só em chamados_hora")
     .when(c == h, "iguais")
     .otherwise("diferentes"),
)
print("\n=== id_cliente: chamados x chamados_hora ===")
display(df_comp.groupBy("status_id_cliente").count().orderBy(F.desc("count")))
mostrar(
    df_comp.filter(F.col("status_id_cliente") == "diferentes")
           .select("id_chamado", "id_cliente", "id_cliente_hora"),
    "Chamados com id_cliente diferente entre as duas tabelas",
)

### 7.2 Chamados sem data de abertura

Separa os chamados sem nenhum registro em `tb_chamados_hora` dos que têm registro, mas sem data de abertura.

In [0]:
sem_registro  = df_comp.filter(F.col("_tem_hora").isNull())
abertura_nula = df_comp.filter(F.col("_tem_hora") & F.col("abertura_data").isNull())

print("\n=== chamados sem data de abertura ===")
print(f"Sem nenhuma linha em chamados_hora: {sem_registro.count()}")
print(f"Com linha em chamados_hora, mas abertura_data nula: {abertura_nula.count()}")
mostrar(sem_registro.select("id_chamado", "id_cliente", "motivo", "canal"),
        "Chamados sem registro de horário")
mostrar(abertura_nula, "Chamados com registro de horário, mas abertura_data nula")

df_chamados_com_hora = df_chamados_com_hora.drop("_tem_hora", "id_cliente_hora")

## 8. dim_status

Combina `resolvido` e abandono em três status: resolvido, não resolvido com atendimento e abandonado na espera. Como regra de negócio, `resolvido` nulo é tratado como não resolvido. Um chamado é considerado abandonado quando foi aberto, não teve início de atendimento e não foi resolvido.

In [0]:
# ---------- regra de negócio: resolvido nulo = não resolvido ----------
df_chamados_com_hora = df_chamados_com_hora.withColumn(
    "resolvido", F.coalesce(F.col("resolvido"), F.lit(False))
)

# ---------- flag de abandono ----------
# abandono: o chamado foi aberto, não teve início de atendimento e não foi resolvido
df_chamados_com_hora = df_chamados_com_hora.withColumn(
    "abandono",
    (~F.col("resolvido"))
    & F.col("abertura_data").isNotNull()
    & F.col("inicio_data").isNull(),
)

print("Distribuição de resolvido x abandono nos chamados:")
display(df_chamados_com_hora.groupBy("resolvido", "abandono").count().orderBy("resolvido", "abandono"))

# ---------- dimensão ----------
df_dim_status = spark.createDataFrame(
    [
        (1, True,  False),     # resolvido
        (2, False, False),     # não resolvido, mas foi atendido
        (3, False, True),      # abandonado na espera
    ],
    "SK_status_chamado int, resolvido boolean, abandono boolean",
).orderBy("SK_status_chamado")

display(df_dim_status)

df_dim_status.write.mode("overwrite").saveAsTable(f"{gold_schema}.dim_status")

## 9. fato_chamado

### 9.1 Leitura das dimensões

In [0]:
df_dim_cliente   = spark.read.table(f"{gold_schema}.dim_cliente")
df_dim_atendente = spark.read.table(f"{gold_schema}.dim_atendente")
df_dim_canal     = spark.read.table(f"{gold_schema}.dim_canal")
df_dim_motivo    = spark.read.table(f"{gold_schema}.dim_motivo")
df_dim_status    = spark.read.table(f"{gold_schema}.dim_status")
df_dim_data      = spark.read.table(f"{gold_schema}.dim_data")
df_dim_hora      = spark.read.table(f"{gold_schema}.dim_hora")

### 9.2 Custo e nota

Acrescenta o custo de `silver.tb_custos` e a nota de `silver.tb_pesquisa_satisfacao`, avisando se algum `id_chamado` repetido duplicaria chamados no join.

In [0]:
df_custos   = spark.read.table(f"{silver_schema}.tb_custos").select("id_chamado", "custo")
df_pesquisa = spark.read.table(f"{silver_schema}.tb_pesquisa_satisfacao").select("id_chamado", "nota_atendimento")

for nome, df_ in [("tb_custos", df_custos), ("tb_pesquisa_satisfacao", df_pesquisa)]:
    n_rep = (df_.filter(F.col("id_chamado").isNotNull())
                .groupBy("id_chamado").count().filter(F.col("count") > 1).count())
    if n_rep > 0:
        print(f"ATENÇÃO: {n_rep} id_chamado repetidos em {nome} -> o join vai duplicar chamados")

df_chamados_completo = (
    df_chamados_com_hora
    .join(df_custos, "id_chamado", "left")
    .join(df_pesquisa, "id_chamado", "left")
)

### 9.3 Chaves das dimensões

Cada dimensão entra apenas com a chave de ligação e a SK, renomeada para FK. As dimensões de data e de hora entram uma vez para cada evento: abertura, início e fim do atendimento.

In [0]:
# cada dimensão entra só com a chave de ligação (com o nome da coluna do chamado) e a SK (renomeada para FK)
df_chamados_completo = (
    df_chamados_completo
    .join(df_dim_cliente.select("id_cliente", F.col("SK_cliente").alias("FK_cliente")),
          "id_cliente", "left")
    .join(df_dim_atendente.select(F.col("id_atendente").cast("string").alias("id_atendente"),
                                  F.col("SK_atendente").alias("FK_atendente")),
          "id_atendente", "left")
    .join(df_dim_canal.select(F.col("nome").alias("canal"), F.col("SK_canal").alias("FK_canal")),
          "canal", "left")
    .join(df_dim_motivo.select(F.col("nome").alias("motivo"), F.col("SK_motivo").alias("FK_motivo")),
          "motivo", "left")
    .join(df_dim_status.select("resolvido", "abandono", F.col("SK_status_chamado").alias("FK_status_chamado")),
          ["resolvido", "abandono"], "left")
)

# data e hora: a mesma dimensão entra uma vez para cada evento
EVENTOS = {"abertura": "abertura", "inicio": "inicio_atendimento", "fim": "fim_atendimento"}

for p, suf in EVENTOS.items():
    df_chamados_completo = (
        df_chamados_completo
        .join(df_dim_data.select(F.col("data").alias(f"{p}_data"),
                                 F.col("SK_data").alias(f"FK_data_{suf}")),
              f"{p}_data", "left")
        .join(df_dim_hora.select(F.col("hora").alias(f"{p}_hora"),
                                 F.col("minuto").alias(f"{p}_minuto"),
                                 F.col("SK_hora").alias(f"FK_hora_{suf}")),
              [f"{p}_hora", f"{p}_minuto"], "left")
    )

fks_data = [f"FK_data_{s}" for s in EVENTOS.values()]
fks_hora = [f"FK_hora_{s}" for s in EVENTOS.values()]

### 9.4 Verificações

Confere se os joins duplicaram linhas e lista os valores sem correspondência em cada dimensão. Em seguida, as FKs sem correspondência passam a apontar para o membro -1.

In [0]:
print("=== VERIFICAÇÕES DA FATO ===")
n_base = df_chamados_com_hora.count()
n_fato = df_chamados_completo.count()
print(f"Chamados: {n_base} | linhas após os joins: {n_fato} -> "
      + ("ok" if n_base == n_fato else "ATENÇÃO: algum join duplicou linhas"))

# chave natural preenchida, mas sem par na dimensão
LIGACOES = {
    "FK_cliente": "id_cliente", "FK_atendente": "id_atendente", "FK_canal": "canal",
    "FK_motivo": "motivo", "FK_status_chamado": "resolvido",
    **{f"FK_data_{s}": f"{p}_data" for p, s in EVENTOS.items()},
    **{f"FK_hora_{s}": f"{p}_hora" for p, s in EVENTOS.items()},
}
for fk, natural in LIGACOES.items():
    mostrar(
        df_chamados_completo.filter(F.col(natural).isNotNull() & F.col(fk).isNull())
                            .groupBy(natural).count().orderBy(F.desc("count")),
        f"{natural}: valores sem correspondência na dimensão ({fk} ficou nulo)",
    )

# sem correspondência (id/nome nulo ou não encontrado) aponta para o membro "Não Informado" (-1)
FKS_COM_MEMBRO_DESCONHECIDO = fks_data + fks_hora + ["FK_cliente", "FK_atendente", "FK_canal", "FK_motivo"]

df_chamados_completo = df_chamados_completo.withColumns(
    {fk: F.coalesce(F.col(fk), F.lit(-1)) for fk in FKS_COM_MEMBRO_DESCONHECIDO}
)

### 9.5 Gravação da fato

O tempo de espera usa o valor informado. Na falta dele, usa a diferença entre início e abertura, quando não é negativa, e por último 0. Tempo de atendimento e custo sem valor ficam 0. A nota sem avaliação fica nula, para não entrar na média.

In [0]:
# espera calculada pela diferença início - abertura (precisão de minuto); só quando não é negativa
espera_calculada = (
    F.datediff("inicio_data", "abertura_data") * 86400
    + ((F.col("inicio_hora") * 60 + F.col("inicio_minuto"))
       - (F.col("abertura_hora") * 60 + F.col("abertura_minuto"))) * 60
)
espera_calculada = F.when(espera_calculada >= 0, espera_calculada)

df_fato_chamado = df_chamados_completo.select(
    "id_chamado",
    "FK_cliente",
    "FK_atendente",
    "FK_canal",
    "FK_motivo",
    *fks_data,
    *fks_hora,
    "FK_status_chamado",
    # informado -> calculado pelos horários -> 0
    F.coalesce(F.col("tempo_espera"), espera_calculada, F.lit(0)).alias("tempo_espera"),
    F.coalesce(F.col("tempo_atendimento"), F.lit(0)).alias("tempo_atendimento"),
    F.coalesce(F.col("custo"), F.lit(0)).alias("custo"),
    # nota nula = chamado sem avaliação: fica nula para não entrar na média
    F.col("nota_atendimento").alias("nota_avaliacao"),
)

display(df_fato_chamado)

df_fato_chamado.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{gold_schema}.fato_chamado")

## 10. Validações

Consultas que conferem a integridade do modelo e a coerência dos dados da fato. Cada uma indica o resultado esperado.

In [0]:
%sql
USE CATALOG vcredit_medallion;

### Validação 1a: Tempo de espera nulo por canal

Mostra em quais canais o `tempo_espera` é nulo e quanto a média muda conforme o tratamento do nulo.

In [0]:
%sql
SELECT c.nome AS canal,
       COUNT(*)                                                   AS chamados,
       COUNT_IF(f.tempo_espera IS NULL)                           AS espera_nula,
       ROUND(100 * COUNT_IF(f.tempo_espera IS NULL) / COUNT(*), 2) AS pct_espera_nula,
       ROUND(AVG(f.tempo_espera), 1)                              AS media_ignorando_nulos,
       ROUND(AVG(COALESCE(f.tempo_espera, 0)), 1)                 AS media_nulo_como_zero
FROM gold.fato_chamado f
JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
GROUP BY c.nome
ORDER BY pct_espera_nula DESC;

### Validação 1b: Espera nula e início igual à abertura

Verifica se, quando `tempo_espera` é nulo, o início do atendimento coincide com a abertura. Se coincide, a espera é zero de verdade, e não um dado faltante.

In [0]:
%sql
--     Se sim, é espera zero de verdade, e não dado faltante.
SELECT c.nome AS canal,
       CASE
         WHEN f.FK_data_abertura = -1 OR f.FK_hora_abertura = -1                     THEN 'abertura não informada'
         WHEN f.FK_data_inicio_atendimento = -1 OR f.FK_hora_inicio_atendimento = -1 THEN 'início não informado'
         WHEN f.FK_data_inicio_atendimento = f.FK_data_abertura
          AND f.FK_hora_inicio_atendimento = f.FK_hora_abertura                      THEN 'início = abertura'
         ELSE 'início diferente da abertura'
       END AS situacao,
       COUNT(*) AS chamados
FROM gold.fato_chamado f
JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
WHERE f.tempo_espera IS NULL
GROUP BY 1, 2
ORDER BY 1, 3 DESC;

### Validação 2a: Códigos 0, 99 e 999 de atendente

Mostra se os códigos 0, 99 e 999 em `id_atendente` se concentram em algum canal.

In [0]:
%sql
SELECT id_atendente, canal, COUNT(*) AS chamados
FROM silver.tb_chamados
WHERE id_atendente IN ('0', '99', '999')
GROUP BY 1, 2
ORDER BY 1, 3 DESC;

### Validação 2b: Atendente por canal

Para cada canal, conta os chamados com atendente real, com código 0, 99 ou 999 e sem atendente.

In [0]:
%sql
SELECT ch.canal,
       COUNT(*)                                       AS chamados,
       COUNT_IF(a.SK_atendente IS NOT NULL)           AS atendente_real,
       COUNT_IF(ch.id_atendente IN ('0', '99', '999')) AS codigo_0_99_999,
       COUNT_IF(ch.id_atendente IS NULL)              AS sem_atendente
FROM silver.tb_chamados ch
LEFT JOIN gold.dim_atendente a ON ch.id_atendente = CAST(a.id_atendente AS STRING)
GROUP BY ch.canal
ORDER BY ch.canal;

### Validação 3: FKs sem correspondência

Verifica se toda FK da fato existe na dimensão correspondente. FK nula também conta como órfã. O esperado é 0 em todas.

In [0]:
%sql
SELECT 'FK_cliente' AS fk, COUNT(*) AS orfas FROM gold.fato_chamado f LEFT ANTI JOIN gold.dim_cliente d ON f.FK_cliente = d.SK_cliente
UNION ALL SELECT 'FK_atendente', COUNT(*)  FROM gold.fato_chamado f LEFT ANTI JOIN gold.dim_atendente d ON f.FK_atendente = d.SK_atendente
UNION ALL SELECT 'FK_canal', COUNT(*)      FROM gold.fato_chamado f LEFT ANTI JOIN gold.dim_canal d     ON f.FK_canal = d.SK_canal
UNION ALL SELECT 'FK_motivo', COUNT(*)     FROM gold.fato_chamado f LEFT ANTI JOIN gold.dim_motivo d    ON f.FK_motivo = d.SK_motivo
UNION ALL SELECT 'FK_status_chamado', COUNT(*) FROM gold.fato_chamado f LEFT ANTI JOIN gold.dim_status d ON f.FK_status_chamado = d.SK_status_chamado
UNION ALL SELECT 'FK_data_abertura', COUNT(*)           FROM gold.fato_chamado f LEFT ANTI JOIN gold.dim_data d ON f.FK_data_abertura = d.SK_data
UNION ALL SELECT 'FK_data_inicio_atendimento', COUNT(*) FROM gold.fato_chamado f LEFT ANTI JOIN gold.dim_data d ON f.FK_data_inicio_atendimento = d.SK_data
UNION ALL SELECT 'FK_data_fim_atendimento', COUNT(*)    FROM gold.fato_chamado f LEFT ANTI JOIN gold.dim_data d ON f.FK_data_fim_atendimento = d.SK_data
UNION ALL SELECT 'FK_hora_abertura', COUNT(*)           FROM gold.fato_chamado f LEFT ANTI JOIN gold.dim_hora d ON f.FK_hora_abertura = d.SK_hora
UNION ALL SELECT 'FK_hora_inicio_atendimento', COUNT(*) FROM gold.fato_chamado f LEFT ANTI JOIN gold.dim_hora d ON f.FK_hora_inicio_atendimento = d.SK_hora
UNION ALL SELECT 'FK_hora_fim_atendimento', COUNT(*)    FROM gold.fato_chamado f LEFT ANTI JOIN gold.dim_hora d ON f.FK_hora_fim_atendimento = d.SK_hora;

### Validação 4a: Uma linha por chamado

Verifica se a fato tem exatamente uma linha por chamado. O esperado é `linhas` igual a `ids_distintos` e nenhum id nulo.

In [0]:
%sql
SELECT COUNT(*) AS linhas,
       COUNT(DISTINCT id_chamado) AS ids_distintos,
       COUNT_IF(id_chamado IS NULL) AS ids_nulos
FROM gold.fato_chamado;

### Validação 4b: Unicidade nas dimensões

Verifica se as dimensões têm SK e chave natural únicas.

In [0]:
%sql
-- Esperado: sks = linhas em todas. chaves_naturais = linhas - 1 em cliente, atendente, data e hora
-- (o membro -1 tem chave nula) e = linhas em canal e motivo (o -1 tem nome 'Não Informado').
SELECT 'dim_cliente' AS dim, COUNT(*) AS linhas, COUNT(DISTINCT SK_cliente) AS sks, COUNT(DISTINCT id_cliente) AS chaves_naturais FROM gold.dim_cliente
UNION ALL SELECT 'dim_atendente', COUNT(*), COUNT(DISTINCT SK_atendente), COUNT(DISTINCT id_atendente) FROM gold.dim_atendente
UNION ALL SELECT 'dim_canal',     COUNT(*), COUNT(DISTINCT SK_canal),     COUNT(DISTINCT nome)         FROM gold.dim_canal
UNION ALL SELECT 'dim_motivo',    COUNT(*), COUNT(DISTINCT SK_motivo),    COUNT(DISTINCT nome)         FROM gold.dim_motivo
UNION ALL SELECT 'dim_data',      COUNT(*), COUNT(DISTINCT SK_data),      COUNT(DISTINCT data)         FROM gold.dim_data
UNION ALL SELECT 'dim_hora',      COUNT(*), COUNT(DISTINCT SK_hora),      COUNT(DISTINCT hora, minuto) FROM gold.dim_hora;

### Validação 5: Chamados apontando para o membro -1

Percentual de chamados que apontam para o membro "Não Informado" em cada FK. Mede o impacto da qualidade do dado nos relatórios.

In [0]:
%sql
SELECT ROUND(100 * COUNT_IF(FK_cliente = -1) / COUNT(*), 2)                 AS pct_cliente,
       ROUND(100 * COUNT_IF(FK_atendente = -1) / COUNT(*), 2)               AS pct_atendente,
       ROUND(100 * COUNT_IF(FK_canal = -1) / COUNT(*), 2)                   AS pct_canal,
       ROUND(100 * COUNT_IF(FK_motivo = -1) / COUNT(*), 2)                  AS pct_motivo,
       ROUND(100 * COUNT_IF(FK_data_abertura = -1) / COUNT(*), 2)           AS pct_data_abertura,
       ROUND(100 * COUNT_IF(FK_data_inicio_atendimento = -1) / COUNT(*), 2) AS pct_data_inicio,
       ROUND(100 * COUNT_IF(FK_data_fim_atendimento = -1) / COUNT(*), 2)    AS pct_data_fim,
       ROUND(100 * COUNT_IF(FK_hora_abertura = -1) / COUNT(*), 2)           AS pct_hora_abertura
FROM gold.fato_chamado;

### Validação 6: Ordem dos eventos

Verifica se abertura, início e fim estão em ordem cronológica. O esperado é 0 nas duas primeiras colunas.

In [0]:
%sql
-- Como SK_data = yyyyMMdd e SK_hora = hhmm, SK_data * 10000 + SK_hora = yyyyMMddhhmm, um número que ordena no tempo.
WITH t AS (
  SELECT CAST(FK_data_abertura AS BIGINT) * 10000 + FK_hora_abertura                     AS abertura,
         CAST(FK_data_inicio_atendimento AS BIGINT) * 10000 + FK_hora_inicio_atendimento AS inicio,
         CAST(FK_data_fim_atendimento AS BIGINT) * 10000 + FK_hora_fim_atendimento       AS fim
  FROM gold.fato_chamado
  WHERE -1 NOT IN (FK_data_abertura, FK_hora_abertura, FK_data_inicio_atendimento,
                   FK_hora_inicio_atendimento, FK_data_fim_atendimento, FK_hora_fim_atendimento)
)
SELECT COUNT_IF(inicio < abertura) AS inicio_antes_da_abertura,
       COUNT_IF(fim < inicio)      AS fim_antes_do_inicio,
       COUNT(*)                    AS chamados_com_os_tres_eventos
FROM t;

### Validação 7: Coerência dos chamados abandonados

Um chamado abandonado não deveria ter início, atendente nem tempo de atendimento. O esperado, na linha de abandono, é que as três últimas colunas fiquem perto de 0.

In [0]:
%sql
-- Esperado na linha abandono = true: as três últimas colunas perto de 0.
SELECT s.resolvido, s.abandono,
       COUNT(*)                                     AS chamados,
       COUNT_IF(f.FK_data_inicio_atendimento <> -1) AS com_inicio,
       COUNT_IF(f.FK_atendente <> -1)               AS com_atendente_real,
       COUNT_IF(f.tempo_atendimento > 0)            AS com_tempo_atendimento
FROM gold.fato_chamado f
JOIN gold.dim_status s ON f.FK_status_chamado = s.SK_status_chamado
GROUP BY s.resolvido, s.abandono
ORDER BY s.resolvido, s.abandono;

### Validação 8: Espera informada e espera calculada

Compara o `tempo_espera` informado com a diferença entre início e abertura, com tolerância de 60 segundos porque a hora foi guardada só até o minuto.

In [0]:
%sql
-- Tolerância de 60 s, porque a hora foi guardada só até o minuto.
WITH t AS (
  SELECT f.tempo_espera,
         make_timestamp(da.ano, da.mes, da.dia, ha.hora, ha.minuto, 0) AS ts_abertura,
         make_timestamp(di.ano, di.mes, di.dia, hi.hora, hi.minuto, 0) AS ts_inicio
  FROM gold.fato_chamado f
  JOIN gold.dim_data da ON f.FK_data_abertura = da.SK_data
  JOIN gold.dim_hora ha ON f.FK_hora_abertura = ha.SK_hora
  JOIN gold.dim_data di ON f.FK_data_inicio_atendimento = di.SK_data
  JOIN gold.dim_hora hi ON f.FK_hora_inicio_atendimento = hi.SK_hora
  WHERE f.tempo_espera IS NOT NULL
    AND -1 NOT IN (f.FK_data_abertura, f.FK_hora_abertura, f.FK_data_inicio_atendimento, f.FK_hora_inicio_atendimento)
)
SELECT CASE WHEN ABS((unix_timestamp(ts_inicio) - unix_timestamp(ts_abertura)) - tempo_espera) < 60
            THEN 'bate (±1 min)' ELSE 'diverge' END AS comparacao,
       COUNT(*) AS chamados
FROM t
GROUP BY 1;

### Validação 9a: Atendimento fora do período do atendente

Conta chamados atendidos antes da admissão ou depois do desligamento do atendente. O esperado é 0 nas duas colunas.

In [0]:
%sql
SELECT COUNT(*)                                 AS chamados_com_atendente,
       COUNT_IF(d.data < a.data_admissao)       AS antes_da_admissao,
       COUNT_IF(d.data > a.data_desligamento)   AS depois_do_desligamento
FROM gold.fato_chamado f
JOIN gold.dim_atendente a ON f.FK_atendente = a.SK_atendente
JOIN gold.dim_data d      ON f.FK_data_inicio_atendimento = d.SK_data
WHERE f.FK_atendente <> -1 AND f.FK_data_inicio_atendimento <> -1;

### Validação 9b: Chamado antes do cadastro do cliente

Conta chamados abertos antes da data de cadastro do cliente. O esperado é 0.

In [0]:
%sql
SELECT COUNT(*)                          AS chamados_com_cliente,
       COUNT_IF(d.data < c.data_cadastro) AS antes_do_cadastro
FROM gold.fato_chamado f
JOIN gold.dim_cliente c ON f.FK_cliente = c.SK_cliente
JOIN gold.dim_data d    ON f.FK_data_abertura = d.SK_data
WHERE f.FK_cliente <> -1 AND f.FK_data_abertura <> -1;

### Validação 10a: Faixas das métricas

Mínimo, mediana, p99 e máximo de tempos e custo, para identificar negativos, máximos absurdos e caudas longas.

In [0]:
%sql
SELECT 'tempo_espera' AS metrica, COUNT_IF(tempo_espera IS NULL) AS nulos,
       MIN(CAST(tempo_espera AS DOUBLE)) AS minimo, percentile_approx(CAST(tempo_espera AS DOUBLE), 0.5) AS mediana,
       percentile_approx(CAST(tempo_espera AS DOUBLE), 0.99) AS p99, MAX(CAST(tempo_espera AS DOUBLE)) AS maximo
FROM gold.fato_chamado
UNION ALL
SELECT 'tempo_atendimento', COUNT_IF(tempo_atendimento IS NULL),
       MIN(CAST(tempo_atendimento AS DOUBLE)), percentile_approx(CAST(tempo_atendimento AS DOUBLE), 0.5),
       percentile_approx(CAST(tempo_atendimento AS DOUBLE), 0.99), MAX(CAST(tempo_atendimento AS DOUBLE))
FROM gold.fato_chamado
UNION ALL
SELECT 'custo', COUNT_IF(custo IS NULL),
       MIN(CAST(custo AS DOUBLE)), percentile_approx(CAST(custo AS DOUBLE), 0.5),
       percentile_approx(CAST(custo AS DOUBLE), 0.99), MAX(CAST(custo AS DOUBLE))
FROM gold.fato_chamado;

### Validação 10b: Distribuição das notas

As notas devem estar na escala de 1 a 5 da pesquisa.

In [0]:
%sql
SELECT nota_avaliacao, COUNT(*) AS chamados
FROM gold.fato_chamado
GROUP BY nota_avaliacao
ORDER BY nota_avaliacao;